# PyTorch: Collaborative Filtering

In [ ]:
from fastai.collab import *
from fastai.tabular.all import *

## Prepare Dataset

In [ ]:
path = untar_data(URLs.ML_100k)

In [ ]:
ratings = pd.read_csv(path/"u.data", delimiter="\t", header=None,
                      names=["user","movie","rating","timestamp"])
ratings.head()

In [ ]:
movies = pd.read_csv(path/"u.item",  delimiter="|", encoding="latin-1",
                     usecols=(0,1), names=["movie","title"], header=None)
movies.head()

In [ ]:
ratings = ratings.merge(movies)
ratings.head()

In [ ]:
dls = CollabDataLoaders.from_df(ratings, item_name="title", bs=64)
dls.show_batch()

## Approach 1: Probabilistic Matrix Factorization (PMF)
(Using dot product model)

In [ ]:
# Get the number of users
n_users = len(dls.classes["user"])
# Get the number of films
n_films = len(dls.classes['title'])
n_users, n_films

In [ ]:
class DotProduct(Module):
    def __init__(self, n_users, n_films, n_factors, y_range=(0,5.5)):
        # Create a matrix with n_factors for users
        self.u_factors = Embedding(n_users, n_factors)
        self.u_bias = Embedding(n_users, 1)
        # Create a matrix with n_factors for films
        self.f_factors = Embedding(n_films, n_factors)
        self.f_bias = Embedding(n_films, 1)
        # Specify the range for "rating"
        self.y_range = y_range
        
    def forward(self, x):
        f_users = self.u_factors(x[:,0])
        b_users = self.u_bias(x[:,0])
        f_films = self.f_factors(x[:,1])
        b_films = self.f_bias(x[:,1])
        # Predict rating and bound to the range
        r = (f_users*f_films).sum(dim=1, keepdim=True) + b_users + b_films
        return sigmoid_range(r, *self.y_range)
  


In [ ]:
model = DotProduct(n_users, n_films, 50)

In [ ]:
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.1)

In [ ]:
# Find a movies nearly identiacal to other using "CosineSimilarity"
f_factors = learn.model.f_factors.weight
idx = dls.classes["title"].o2i["Silence of the Lambs, The (1991)"]
distances = nn.CosineSimilarity(dim=1)(f_factors, f_factors[idx][None])
idx = distances.argsort(descending=True)[1]
dls.classes["title"][idx]

## Approach 2: Deep Learning

In [ ]:
# Get recommended sizes for embedding matrices 
embs = get_emb_sz(dls)
embs

In [ ]:
class CollabNN(Module):
    def __init__(self, user_sz, item_sz, y_range=(0,5.5), n_act=100):
        self.user_factors = Embedding(*user_sz)
        self.item_factors = Embedding(*item_sz)
        self.layers = nn.Sequential(
            nn.Linear(user_sz[1]+item_sz[1], n_act),
            nn.ReLU(),
            nn.Linear(n_act, 1))
        self.y_range = y_range
        
    def forward(self, x):
        embs = self.user_factors(x[:,0]),self.item_factors(x[:,1])
        x = self.layers(torch.cat(embs, dim=1))
        return sigmoid_range(x, *self.y_range)

In [ ]:
model = CollabNN(*embs)

In [ ]:
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.01)